# Gold: Price by temperature band
Aggregates NO2 electricity prices across 5-degree temperature bands using joined Silver data, calculates descriptive price statistics, and persists the results using upsert and scoped delete MERGE.

In [0]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

dbutils.widgets.text("run_id", "")
RUN_ID = dbutils.widgets.get("run_id") or str(uuid.uuid4())

In [0]:
# Load joined silver data for NO2, filtering out rows without temperature
source_df = (
    spark.table(f"{CATALOG}.silver.weather_price_hourly")
    .filter(
        (F.col("price_area") == "NO2")
        & F.col("air_temperature_c").isNotNull()
    )
)

if source_df.count() == 0:
    raise ValueError("Gold source is empty")

In [0]:
# Aggregate metrics per 5-degree temperature band
gold_df = (
    source_df
    .withColumn(
        "temperature_band_order",
        F.floor(F.col("air_temperature_c") / F.lit(5.0)).cast("int")
    )
    .withColumn(
        "temperature_band",
        F.concat(
            (F.col("temperature_band_order") * 5).cast("string"),
            F.lit(" to "),
            ((F.col("temperature_band_order") * 5) + 5).cast("string"),
            F.lit(" °C")
        )
    )
    .groupBy(
        "price_area", "temperature_band_order", "temperature_band"
    )
    .agg(
        F.count("*").alias("observation_count"),
        F.round(F.avg("nok_per_kwh"), 4)
        .alias("average_price_nok_per_kwh"),
        F.round(
            F.expr("percentile_approx(nok_per_kwh, 0.5)"), 4
        ).alias("median_price_nok_per_kwh"),
        F.round(F.min("nok_per_kwh"), 4)
        .alias("minimum_price_nok_per_kwh"),
        F.round(F.max("nok_per_kwh"), 4)
        .alias("maximum_price_nok_per_kwh"),
        F.round(F.avg("wind_speed_mps"), 2)
        .alias("average_wind_speed_mps")
    )
    .withColumn("updated_at_utc", F.current_timestamp())
)

if gold_df.count() == 0:
    raise ValueError("Gold output is empty")

In [0]:
# Upsert MERGE with scoped delete for NO2
(
    DeltaTable.forName(
        spark, f"{CATALOG}.gold.price_by_temperature_band"
    ).alias("target")
    .merge(
        gold_df.alias("source"),
        """
        target.price_area = source.price_area
        AND target.temperature_band_order = source.temperature_band_order
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete("target.price_area = 'NO2'")
    .execute()
)

# Log data quality check for Gold
check = [(
    RUN_ID, "gold", "gold_output_nonempty", "ERROR", True,
    float(gold_df.count()), "Gold contains temperature-band rows",
    datetime.now(timezone.utc).replace(tzinfo=None)
)]

dq_df = spark.createDataFrame(check, """
    run_id string, layer string, check_name string, severity string,
    passed boolean, actual_value double, message string,
    checked_at_utc timestamp
""")

(
    DeltaTable.forName(
        spark, f"{CATALOG}.ops.data_quality_results"
    ).alias("target")
    .merge(
        dq_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.layer = source.layer
        AND target.check_name = source.check_name
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"Gold analysis completed. run_id={RUN_ID}")
display(gold_df.orderBy("temperature_band_order"))